# Tabular ML Basics

## Learning goals
- Understand what tabular supervised learning looks like in practice.
- Define **features** and a **target** clearly.
- Build train / validation / test splits.
- Preprocess missing and categorical values.
- Train and compare multiple baseline models.
- Evaluate with metrics that match the task.


In [ ]:
# Notebook-specific environment setup.
# In Colab, many of these packages already exist, but we install missing ones
# so this notebook remains self-contained when opened on its own.

import importlib.util
import subprocess
import sys


REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
}

for import_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"{import_name} is already installed.")


## Why start with tabular data?

Tabular data is still one of the most common formats in practical machine learning:
- one row per person, event, or item
- one column per measured variable
- one target column to predict

We will use the **Titanic** dataset and predict whether a passenger survived.

In [ ]:
# Import the tools we need for data loading, visualization, preprocessing, and modeling.
# We keep the imports grouped so learners can see the main building blocks in one place.

import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# A fixed seed makes the train / validation / test split reproducible.
# This means the class can get similar results when they rerun the notebook.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Set a clean plotting style for simple visuals.
sns.set_theme(style="whitegrid")


## Important lines in the previous cell

- `SEED = 42`: chooses a fixed random seed so the same random split can be repeated.
- `random.seed(SEED)` and `np.random.seed(SEED)`: apply that seed to Python and NumPy randomness.
- `from sklearn.model_selection import train_test_split`: imports the tool we use to hold out data for honest evaluation.
- `Pipeline`, `ColumnTransformer`, `OneHotEncoder`, and `SimpleImputer`: these are the pieces that let us build a clean preprocessing + model workflow.


In [ ]:
# Load a small public CSV directly from a raw GitHub URL.
# This avoids local file paths and works well in Colab.

DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

try:
    df = pd.read_csv(DATA_URL)
except Exception as exc:
    raise RuntimeError(
        "Could not download the Titanic dataset. "
        "Please check your Colab internet connection and rerun the cell."
    ) from exc

print(f"Loaded dataset shape: {df.shape}")
df.head()


## What the previous lines do

- `DATA_URL = ...`: stores the dataset location as a variable so it can be reused or changed easily.
- `pd.read_csv(DATA_URL)`: downloads the CSV and converts it into a pandas DataFrame.
- `try ... except`: gives a clearer error if the download fails.
- `df.head()`: shows the first few rows so we can inspect the table before modeling.


## Features and target

The **target** is the column we want to predict:
- `Survived`

The **features** are the inputs we will give the model:
- `Pclass`
- `Sex`
- `Age`
- `SibSp`
- `Parch`
- `Fare`
- `Embarked`

We leave out columns like `Name`, `Ticket`, and `Cabin` for simplicity.


In [ ]:
# Keep only the columns we want to teach with.
# This reduces noise and makes every later step easier to explain.

selected_columns = [
    "Survived",
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
]

# .copy() avoids warnings and makes it clear that we are working on a new DataFrame.
df = df[selected_columns].copy()

print("Columns used in this notebook:")
print(df.columns.tolist())
print(f"Filtered dataset shape: {df.shape}")

df.head()


In [ ]:
# Basic EDA: inspect shape, class balance, and summary statistics.
# This helps us understand the problem before we build any model.

print("Number of rows and columns:")
print(df.shape)

print("\nClass distribution:")
print(df["Survived"].value_counts(normalize=True).rename("proportion"))

display(df.describe(include="all").T)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Survived")
plt.title("Class distribution: survival outcome")
plt.xlabel("Survived")
plt.ylabel("Count")
plt.show()


In [ ]:
# Two simple plots:
# 1. Survival rate by sex
# 2. Age distribution colored by survival outcome

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

survival_by_sex = (
    df.groupby("Sex")["Survived"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

sns.barplot(data=survival_by_sex, x="Sex", y="Survived", ax=axes[0])
axes[0].set_title("Average survival rate by sex")
axes[0].set_ylabel("Mean survival")

sns.histplot(
    data=df,
    x="Age",
    hue="Survived",
    bins=25,
    kde=True,
    element="step",
    ax=axes[1],
)
axes[1].set_title("Age distribution by survival outcome")

plt.tight_layout()
plt.show()


## Why this EDA matters

The plots already tell us useful things:
- the classes are not perfectly balanced
- age has missing values
- survival patterns appear to differ across groups

This does **not** mean the model understands the world. It only means the data contains patterns that may be predictive.


In [ ]:
# Split the DataFrame into X (features) and y (target).
# This is a standard convention in machine learning code.

target_column = "Survived"
feature_columns = [column for column in df.columns if column != target_column]

X = df[feature_columns]
y = df[target_column]

print("Feature columns:")
print(feature_columns)
print("\nMissing values per feature:")
print(X.isna().sum())


## Train / validation / test split

We use three splits because they play different roles:
- **Train**: fit the model
- **Validation**: compare models and tune settings
- **Test**: final honest check at the end

If you keep changing choices based on the test set, the test set stops being a fair test.


In [ ]:
# First split: hold out a test set.
# stratify=y keeps the class balance roughly similar in each split.

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

# Second split: carve the remaining data into train and validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,  # 25% of 80% gives 20% of the full dataset.
    stratify=y_train_val,
    random_state=SEED,
)

print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

# Identify numeric and categorical columns.
# We do this because different column types need different preprocessing.
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

# Numeric preprocessing:
# 1. Fill missing numeric values with the median.
# 2. Scale numeric columns so distance-based models like kNN behave better.
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Categorical preprocessing:
# 1. Fill missing categories with the most common value.
# 2. Turn categories into numeric indicator columns.
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# ColumnTransformer applies the numeric pipeline to numeric columns
# and the categorical pipeline to categorical columns.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


def make_pipeline(model):
    # This helper keeps preprocessing and model training together.
    # That makes the workflow cleaner and reduces the risk of data leakage.
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )


## Important lines in the splitting and preprocessing cell

- `train_test_split(..., stratify=y, random_state=SEED)`: makes the split reproducible and keeps class balance more stable.
- `numeric_features = [...]` and `categorical_features = [...]`: tells scikit-learn which columns need which kind of treatment.
- `SimpleImputer(strategy="median")`: replaces missing numeric values with the median of that column.
- `StandardScaler()`: rescales numeric columns so they are more comparable in size.
- `OneHotEncoder(handle_unknown="ignore")`: converts text categories into numeric columns and safely ignores unseen categories later.
- `Pipeline([...])`: chains preprocessing and modeling into a single reusable object.


## Train a few simple models

We will compare three beginner-friendly classifiers:
- **Logistic regression**
- **k-nearest neighbors**
- **Random forest**

This is a good habit. Model choice should be based on evidence, not guesswork.


In [ ]:
# Define a small set of baseline models.
# We intentionally start simple rather than jumping to the most complex option.

models = {
    "Logistic regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "kNN (k=7)": KNeighborsClassifier(n_neighbors=7),
    "Random forest": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
    ),
}

fitted_models = {}
validation_rows = []

for model_name, model in models.items():
    # Build a fresh pipeline for this model.
    pipeline = make_pipeline(model)

    # Fit on the training split only.
    pipeline.fit(X_train, y_train)

    # Predict on the validation split to compare models fairly.
    val_predictions = pipeline.predict(X_val)

    validation_rows.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(y_val, val_predictions),
            "precision": precision_score(y_val, val_predictions, zero_division=0),
            "recall": recall_score(y_val, val_predictions, zero_division=0),
            "f1": f1_score(y_val, val_predictions, zero_division=0),
        }
    )

    fitted_models[model_name] = pipeline

results_df = pd.DataFrame(validation_rows).sort_values(by="f1", ascending=False)
display(results_df.style.format("{:.3f}"))


## What the model comparison cell is doing

- `models = {...}`: stores multiple candidate models in one dictionary so we can loop through them.
- `pipeline.fit(X_train, y_train)`: learns from the training split only.
- `pipeline.predict(X_val)`: makes predictions on unseen validation data.
- `accuracy_score`, `precision_score`, `recall_score`, `f1_score`: compute different views of model quality.
- `sort_values(by="f1", ascending=False)`: ranks the models by validation F1 score.

**Why this matters**

A model can look strong on one metric and weaker on another. The right metric depends on the real-world cost of mistakes.


In [ ]:
# Pick the best model based on validation F1, then evaluate it once on the test set.
# The test set should stay untouched until this point.

best_model_name = results_df.iloc[0]["model"]
best_pipeline = fitted_models[best_model_name]

test_predictions = best_pipeline.predict(X_test)

test_metrics = pd.DataFrame(
    [
        {
            "best_model": best_model_name,
            "accuracy": accuracy_score(y_test, test_predictions),
            "precision": precision_score(y_test, test_predictions, zero_division=0),
            "recall": recall_score(y_test, test_predictions, zero_division=0),
            "f1": f1_score(y_test, test_predictions, zero_division=0),
        }
    ]
)

display(test_metrics.style.format("{:.3f}"))

print("Classification report on the test set:")
print(classification_report(y_test, test_predictions, zero_division=0))

cm = confusion_matrix(y_test, test_predictions)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1]).plot(cmap="Blues")
plt.title(f"Confusion matrix for: {best_model_name}")
plt.show()


## How to interpret the test results

- **Accuracy** answers: what fraction of predictions were correct?
- **Precision** answers: when the model predicts survival, how often is it right?
- **Recall** answers: of the passengers who survived, how many did the model find?
- **F1** balances precision and recall.
- The **confusion matrix** shows where the model is making mistakes.

For some problems, recall matters most. For others, precision matters more. Accuracy alone is often not enough.


In [ ]:
# Example predictions on made-up passengers.
# This makes the abstract model outputs easier to connect to concrete cases.

example_passengers = pd.DataFrame(
    [
        {
            "Pclass": 1,
            "Sex": "female",
            "Age": 28,
            "SibSp": 0,
            "Parch": 0,
            "Fare": 80.0,
            "Embarked": "C",
        },
        {
            "Pclass": 3,
            "Sex": "male",
            "Age": 35,
            "SibSp": 0,
            "Parch": 0,
            "Fare": 7.9,
            "Embarked": "S",
        },
        {
            "Pclass": 2,
            "Sex": "female",
            "Age": 10,
            "SibSp": 1,
            "Parch": 2,
            "Fare": 26.0,
            "Embarked": "S",
        },
    ]
)

example_predictions = best_pipeline.predict(example_passengers)
example_probabilities = best_pipeline.predict_proba(example_passengers)[:, 1]

example_results = example_passengers.copy()
example_results["predicted_survival"] = example_predictions
example_results["predicted_survival_probability"] = example_probabilities

display(example_results.style.format({"predicted_survival_probability": "{:.3f}"}))


## Think about it

- Do these passenger-level predictions feel intuitive?
- Could the model be learning patterns that reflect historical inequality in the dataset?
- Would you trust this model for a real policy decision? Why or why not?


## Exercise 1: change a hyperparameter

Change the value of `custom_k` in the next cell and rerun it.

Questions to discuss:
- What happens when `k` is very small?
- What happens when `k` is larger?
- Does kNN seem more sensitive to hyperparameters than logistic regression?


In [ ]:
# Experiment with a new k value for kNN.
# This is a small but meaningful example of hyperparameter tuning.

custom_k = 3

exercise_knn = make_pipeline(KNeighborsClassifier(n_neighbors=custom_k))
exercise_knn.fit(X_train, y_train)
exercise_predictions = exercise_knn.predict(X_val)

print(f"Validation metrics for kNN with k={custom_k}")
print(f"Accuracy:  {accuracy_score(y_val, exercise_predictions):.3f}")
print(f"Precision: {precision_score(y_val, exercise_predictions, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_val, exercise_predictions, zero_division=0):.3f}")
print(f"F1:        {f1_score(y_val, exercise_predictions, zero_division=0):.3f}")


## Exercise 2: try your own passenger examples

Edit the values in the next cell and rerun it.

Ideas:
- change passenger class
- change sex or age
- leave `Age` missing for one row and see that the pipeline still works
- compare a high-fare and low-fare example


In [ ]:
# Build your own examples.
# Notice that the same preprocessing pipeline handles missing values and categories for you.

custom_passengers = pd.DataFrame(
    [
        {
            "Pclass": 1,
            "Sex": "male",
            "Age": 42,
            "SibSp": 0,
            "Parch": 0,
            "Fare": 120.0,
            "Embarked": "C",
        },
        {
            "Pclass": 3,
            "Sex": "female",
            "Age": np.nan,
            "SibSp": 1,
            "Parch": 1,
            "Fare": 12.5,
            "Embarked": "Q",
        },
    ]
)

custom_predictions = best_pipeline.predict(custom_passengers)
custom_probabilities = best_pipeline.predict_proba(custom_passengers)[:, 1]

custom_results = custom_passengers.copy()
custom_results["predicted_survival"] = custom_predictions
custom_results["predicted_survival_probability"] = custom_probabilities

display(custom_results.style.format({"predicted_survival_probability": "{:.3f}"}))


## Exercise 3: change the split

In the next cell, try a different `alt_test_size`.

Questions to discuss:
- Do the scores move a little or a lot?
- Why can evaluation change when the split changes?
- What does this tell us about limited dataset size?


In [ ]:
# Repeat the split with a different test size.
# This shows that metrics can vary when the data allocation changes.

alt_test_size = 0.30

X_train_val_alt, X_test_alt, y_train_val_alt, y_test_alt = train_test_split(
    X,
    y,
    test_size=alt_test_size,
    stratify=y,
    random_state=SEED,
)

X_train_alt, X_val_alt, y_train_alt, y_val_alt = train_test_split(
    X_train_val_alt,
    y_train_val_alt,
    test_size=0.25,
    stratify=y_train_val_alt,
    random_state=SEED,
)

alt_model = make_pipeline(LogisticRegression(max_iter=1000, random_state=SEED))
alt_model.fit(X_train_alt, y_train_alt)
alt_predictions = alt_model.predict(X_val_alt)

print(f"Alternative validation accuracy: {accuracy_score(y_val_alt, alt_predictions):.3f}")
print(f"Alternative validation F1: {f1_score(y_val_alt, alt_predictions, zero_division=0):.3f}")


## Discussion prompts

- Which metric matters more in this task and why?
- What is one possible source of bias or limitation in the Titanic dataset?
- When is a simple model enough?
- What should we evaluate besides accuracy?